# 데이터셋 파이프라인

동결된 요구사항 데이터셋 `requirements_v0.3.0`을 읽고 무결성을 점검합니다.

## 데이터 단위

```
요구사항 ID 하나 = 데이터 행 하나 = 예측 결과 하나
```

세부 불릿을 독립 행으로 쪼개지 않습니다. 같은 요구사항 안의 조건·예외·제공 주체·책임 범위를
함께 읽어야 판정이 성립하기 때문입니다.

## 이 노트북에서 확인하는 것

1. **재생성 가능성** — 데이터셋이 코드로 다시 만들어지는가 (`data/processed/`는 Git에서 제외됨)
2. **무결성** — 중복 UID, 빈 본문, 문서별 누락이 없는가
3. **분포** — 문서·유형·길이가 어떻게 퍼져 있는가

## 주의

분석 로직은 `scripts/data/eda_requirements.py`가 기준입니다. 이 노트북은 그 함수를 호출해
결과를 보여줄 뿐이며, 여기서 로직을 새로 정의하지 않습니다. 상세 분석은
`03_requirements_eda.ipynb`를 보세요.

In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from scripts.data.eda_requirements import analyze_dataset, load_dataset

# data/processed/는 Git에서 제외되므로 없을 수 있습니다.
# 그때는 build_dataset.py로 원본 Markdown에서 다시 만들면 됩니다.
DATASET = ROOT / 'data/processed/requirements_v0.3.0.jsonl'
if not DATASET.exists():
    raise FileNotFoundError(
        f'{DATASET.name}이 없습니다. 먼저 실행하세요:\n'
        '  python -m scripts.data.build_dataset'
    )

records = load_dataset(str(DATASET))
print(f'{DATASET.name}: {len(records):,}건')
print(f'필드: {sorted(records[0])}')

requirements_v0.3.0.jsonl: 1,024건
필드: ['agency', 'dataset_version', 'document_id', 'domain', 'extraction_adapter', 'normalized_requirement_text', 'raw_requirement_text', 'requirement_id', 'requirement_name', 'requirement_type', 'requirement_uid', 'source_block_index', 'source_file', 'source_location', 'source_requirement_id', 'source_sha256', 'source_table_index']


In [2]:
summary = analyze_dataset(records)

# 데이터셋 규모와 문서별 구성.
# 특정 기관이 데이터를 지나치게 지배하면 문서 단위 평가에서 일반화가 어려워집니다.
print(f"데이터셋 버전: {summary['dataset_version']}")
print(f"총 행 수: {summary['total_records']:,}")
print(f"문서 수: {len(summary['document_counts'])}\n")

print('문서별 행 수')
total = summary['total_records']
for doc, n in sorted(summary['document_counts'].items(), key=lambda x: -x[1]):
    print(f'  {doc:<32} {n:>5}건  ({n / total:>5.1%})')

print('\n본문 길이 (문자)')
for key, value in summary['char_length_stats'].items():
    print(f'  {key:<10} {value}')

데이터셋 버전: requirements_v0.3.0
총 행 수: 1,024
문서 수: 10

문서별 행 수
  mfds_drug_ai_review                192건  (18.8%)
  defense_intelligent_platform       169건  (16.5%)
  kexim_ai_platform                  137건  (13.4%)
  koen_ai_infrastructure             101건  ( 9.9%)
  ccrs_ai_platform                    95건  ( 9.3%)
  kac_ai_work_platform                86건  ( 8.4%)
  incheon_airport_digital_work        78건  ( 7.6%)
  genai_incident_response             67건  ( 6.5%)
  kangwon_land_genai                  50건  ( 4.9%)
  korail_genai_isp_ismp               49건  ( 4.8%)

본문 길이 (문자)
  min        26
  max        4895
  mean       453.35
  median     337.0
  p90        927.8
  p95        1213.6
  p99        2169.54


In [3]:
from collections import Counter

# 무결성 점검. 추출 단계에서 놓친 것이 없는지 확인합니다.
uids = [r['requirement_uid'] for r in records]
duplicates = [uid for uid, n in Counter(uids).items() if n > 1]
empty_text = [r['requirement_uid'] for r in records if not (r.get('raw_requirement_text') or '').strip()]
missing_type = [r['requirement_uid'] for r in records if not r.get('requirement_type')]

checks = [
    ('중복 requirement_uid', duplicates),
    ('빈 본문', empty_text),
    ('요구사항 유형 미지정', missing_type),
]
for name, hits in checks:
    status = 'OK' if not hits else f'{len(hits)}건'
    print(f'{name:<24} {status}')
    for uid in hits[:5]:
        print(f'    - {uid}')

# 유형 미지정은 결함이 아니라 원문에 유형 열이 없는 문서의 특성입니다.
# 예측 타깃이 아니라 입력 메타데이터로만 쓰이므로 그대로 둡니다.
print('\n요구사항 유형 상위 10')
for t, n in Counter(r.get('requirement_type') or '(미지정)' for r in records).most_common(10):
    print(f'  {t:<28} {n:>4}건')

중복 requirement_uid       OK
빈 본문                     OK
요구사항 유형 미지정              OK

요구사항 유형 상위 10
  기능 요구사항                       119건
  프로젝트 관리 요구사항                   64건
  기능                             60건
  보안 요구사항                        51건
  품질 요구사항                        46건
  제약사항                           46건
  데이터 요구사항                       38건
  프로젝트 지원 요구사항                   38건
  기능 요구사항(SFR)                   38건
  테스트 요구사항                       30건
